## load libs, functions and data

In [ ]:
import numpy as np
from tifffile import *
import argparse
import xml.etree.ElementTree as ET
import skimage.io as io
from skimage import exposure
from skimage.filters import gaussian
import matplotlib.pyplot as plt
#from cuml.cluster import KMeans as KMeans
#from cuml.dask.cluster import kmeans as KMeans_dask
import pandas as pd
import collections
import math
import os, psutil
import collections
import gc
import sys
from skimage import transform
from matplotlib import cm
from matplotlib.colors import *
import matplotlib
import random
import numpy as np
#from dask_cuda import LocalCUDACluster
#import cupy as cp
#import cudf
#import dask_cudf
import dask
import time
import scipy
import anndata
import scipy.spatial as scispatial
import json
import scanpy as sc
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from scipy.stats import zscore
import seaborn as sns
import re
import imagecodecs
import h5py
from collections import OrderedDict


#Input all data from file
def tiff_in(inFile):
	with TiffFile(inFile) as tif:
		tif_tags = {}
		for tag in tif.pages[0].tags.values():
			name, value = tag.name, tag.value
			tif_tags[name] = value
			#images = tif.asarray()
		pages = [tif.asarray(),tif_tags]
		shapes=[]
		for series in tif.series:
			shapes.append(series.shape)
	return pages,shapes

def channelNames(inFile):
    markerDict = {}
    iCnt = 0
    with TiffFile(inFile) as tif:
        for page in tif.series[0].pages:
            markerDict[iCnt] = ET.fromstring(page.description).find('Name').text
            iCnt += 1
    return markerDict


def get_colors(num_colors, return_cmap = False):
    import numpy as np
    import colorsys
    from matplotlib.colors import ListedColormap
    colors=[]
    for i in np.arange(0., 360., 360. / num_colors):
        hue = i/360.
        lightness = (50 + np.random.rand() * 10)/100.
        saturation = (90 + np.random.rand() * 10)/100.
        colors.append(colorsys.hls_to_rgb(hue, lightness, saturation))
        
    if return_cmap:
        return(ListedColormap(colors))
    else:
        return colors
    
    
def crop_image(matrix, center_coordinate, buffer_size):
    # Extract dimensions of the original matrix
    height, width = matrix.shape

    # Extract the coordinates of the center
    center_x, center_y = center_coordinate

    print(center_x)
    
    
    # Calculate the bounding box for the cropped image
    start_x = max(0, center_x - buffer_size)
    end_x = min(height, center_x + buffer_size)
    start_y = max(0, center_y - buffer_size)
    end_y = min(width, center_y + buffer_size)

    # Calculate the cropped image
    cropped_image = matrix[start_x:end_x, start_y:end_y]

    return cropped_image



# load in sample all cells adata

In [ ]:
adata = anndata.read_h5ad('../fabian_metadata_allcells.h5ad')

adata.obs

## generate x/y coordinates for non-nucleated features


In [ ]:

def pixels_to_um(units, input_type):
    
    if input_type == 'pixels':
        return(0.3774 * units)
    if input_type == 'um':
        return(units / 0.3774)

print(200 ** 0.5)
print(pixels_to_um(units = 200 ** 0.5, input_type='um')**2)


In [ ]:

# generate x/y coordinates for non-nucleated features

df_dict = {
'image_files':['../image_data/11092021_BaharehNP_3026CtlCorte.qptiff', '../image_data/11162021_Bahareh-3155_EDTA.qptiff', '../image_data/12032021-Bahareh_3280_Alz.qptiff', '../image_data/12252021_Bahareh-3628-Cntr.qptiff', '../image_data/10222021_BaharehNP_1873_CtrlCortex.qptiff', '../image_data/10122021_BaharehNP_1-CtlCortex1796_EDTA_MKT6.qptiff', '../image_data/10122021_BaharehNP_ADCortex_EDTA_3196.qptiff', '../image_data/10202021_BaharehNP_2997-ADCortex_EDTA.qptiff'],
'image_nums':[3026, 3155, 3280, 3628, 1873, 1796, 3196, 2997],
'thresholds':[{'DAPI_1':30, 'Neurofilament':10, 'Vimentin':20, 'Claudin5':24, 'SMA':25, 'A-Beta':30, 'ApoE':60, 'GFAP':30, 'Olig-2':30, 'PSD95':30, 'Synaptophysin':20, 'NeuN':25, 'H2A.x':25, 'CollagenIV':30, 'PCNA':100, 'CD3e':70, 'CD4':80, 'MAP2':20},
             {'DAPI_1':25, 'Neurofilament':20, 'Vimentin':10, 'Claudin-5':60, 'SMA':20, 'a-Beta':18, 'GFAP':33, 
                     'Olig2':26, 'Synaptophysin':20,'ApoE':49, 'IBA1':30, 'H2A.x':15, 'CollagenIV':20, 'PCNA':175, 'MAP2':30}, 
             {'DAPI_1':25, 'Neurofilament':35, 'Vimentin':25, 'Claudin-5':20, 'SMA':20, 'a-Beta':15, 'Olig2':20, 'PSD-95':15, 'ApoE':40, 'NeuN':15, 'H2A.x':35, 'CollagenIV':10, 'PCNA':115, 'MAP2':15},
             {'DAPI_1':30, 'Neurofilament':20, 'Vimentin':30, 'Claudin-5':20, 'SMA':24, 'a-Beta':255, 'Olig2':60, 'PSD-95':20,'ApoE':80, 'CD8':254, 'NeuN':10, 'H2A.x':5, 'CollagenIV':30, 'PCNA':125, 'CD3e':47, 'CD4':50, 'MAP2':20}, {'DAPI_1':30, 'Neurofilament':25, 'Vimentin':20, 'Claudin5':80, 'SMA':20, 'A-Beta':255,'ApoE':60, 'GFAP':35, 'Synaptophysin':10, 'CD8':200, 'NeuN':20, 'H2A.x':55, 'CollagenIV':25, 'PCNA':145, 'CD3e':220, 'MAP2':25}, {'DAPI_1':30, 'Neurofilament':28, 'Vimentin':30, 'Claudin5':43, 'SMA':30, 'A-Beta':30, 'GFAP':30, 'ApoE':60, 'NeuN':25, 'H2A.x':46, 'CollagenIV':30, 'PCNA':103, 'CD3e':116, 'MAP2':40}, {'DAPI_1':35, 'Neurofilament':25, 'Vimentin':40, 'Claudin5':36, 'SMA':30, 'A-Beta':26, 'GFAP':25, 'ApoE':40, 'CD8':250, 'P2Rx7(Park7)':160, 'S100B':80, 'NeuN':30, 'H2A.x':26, 'CollagenIV':22, 'PCNA':230, 'CD3e':250, 'CD4':250, 'MAP2':42}, {'DAPI_1':25, 'Neurofilament':15, 'Vimentin':15, 'Claudin5':18, 'SMA':10, 'A-Beta':15, 'ApoE':40, 'GFAP':20, 'Synaptophysin':27, 'P2Rx7(Park7)':157, 'S100B':13, 'NeuN':15, 'H2A.x':25, 'CollagenIV':10, 'PCNA':117, 'MAP2':30}
],
'nn_structures':[['Claudin5', 'A-Beta', 'ApoE'], ['Claudin-5', 'a-Beta', 'ApoE'], ['Claudin-5', 'a-Beta', 'ApoE'], ['Claudin-5', 'a-Beta', 'ApoE'], ['Claudin5', 'A-Beta', 'ApoE'], ['Claudin5', 'A-Beta', 'ApoE'], ['Claudin5', 'A-Beta', 'ApoE'], ['Claudin5', 'A-Beta', 'ApoE']],
'nn_structure_names':[['vasculature', 'aBeta', 'ApoE']] * 8,
'noise_pixel_thresh':[[100,1404, 1404]] * 8,
'GM_WM_mask':['../GM_WM_Annotations_QP/3026_GM-WM Annotations.geojson', '../GM_WM_Annotations_QP/3155_GM-WM Annotations.geojson', '../GM_WM_Annotations_QP/3280_GM-WM Annotations.geojson', '../GM_WM_Annotations_QP/3628_GM-WM Annotations.geojson', '../GM_WM_Annotations_QP/1873_GM-WM Annotations.geojson', '../GM_WM_Annotations_QP/1796_GM-WM Annotations.geojson', '../GM_WM_Annotations_QP/3196_GM-WM Annotations.geojson', '../GM_WM_Annotations_QP/2997_GM-WM Annotations.geojson']
}

# orig vasc thresh was 400

binar_matrices = {}

image_df = pd.DataFrame(df_dict)

for col in image_df.index:
    row = image_df.iloc[col,:]
    (all_pages,shapes) = tiff_in(row['image_files'])
    markerDict = channelNames(row['image_files'])
    markerDict_reverse = inv_map = {v: k for k, v in markerDict.items()}
    
    
    
    
    for geneind in range(len(row['nn_structures'])):
        
        
        gene = row['nn_structures'][geneind]
        
        

        
        structure_name = row['nn_structure_names'][geneind]
        orig_Img = all_pages[0][markerDict_reverse[gene]]
        print(gene)
        imshow(orig_Img)
        plt.show()
        crop_orig = crop_image(orig_Img, center_coordinate=[1805, 3408], buffer_size=50)
        imshow(crop_orig)
        
        plt.title(str(row['image_nums']) + gene)
        plt.show()
        
        
        tMin = row['thresholds'][gene]
        flat = orig_Img.flatten()
        flat = flat[flat >= tMin]
        a_max = min(np.percentile(flat, q = 99), tMin + 30)
        
        inImg_binar = np.clip(a=orig_Img,a_min=0, a_max = a_max)
        inImg_binar[inImg_binar < tMin] = 0
        inImg_binar[inImg_binar > 0] = 1
        binar_matrices[structure_name + '_' + str(row['image_nums'])] = [inImg_binar, gene, structure_name, structure_name + '_' + str(row['image_nums']), str(row['image_nums']), row['noise_pixel_thresh'][geneind], row['GM_WM_mask']]
        
        
        
        

    
    
    
    # load appropriate image





In [ ]:
import h5py


big_griddles = {}


from skimage import io, color, measure


for image_key in binar_matrices.keys():
    print(image_key)
    
    image = binar_matrices[image_key][0]
    id = binar_matrices[image_key][3]
    structure_name = binar_matrices[image_key][2]
    GM_WM_mask = binar_matrices[image_key][6]
    filenum = binar_matrices[image_key][4]
    
    # Define your original matrix
    original_matrix = image

    # Get the shape of the original matrix
    num_rows, num_cols = original_matrix.shape

    # label connected components 
    labeled_image = measure.label(original_matrix)
    
    # save segmentation
    
    
    import h5py
    import numpy as np
    
    
    
    seg_dict = {}
    seg_dict['matrix'] = np.transpose(labeled_image)
    
    def dict_to_hdf5(group, data):
        for key, value in data.items():
            if isinstance(value, dict):
                subgroup = group.create_group(key)
                dict_to_hdf5(subgroup, value)
            else:
                group[key] = value

    with h5py.File('stored_segmentations/' + structure_name + '_' + filenum + '.h5', 'w') as f:
        dict_to_hdf5(f, seg_dict)
    
    
    
    
    
    
    
    # Find the properties of labeled regions (blobs)
    blob_properties = measure.regionprops(labeled_image)


    # Plot the original image
    plt.imshow(image[2000:4000,2000:4000], cmap='gray')

    # for region in blob_properties:
    #     minr, minc, maxr, maxc = region.bbox
    #     rect = plt.Rectangle((minc, minr), maxc - minc, maxr - minr, fill=False, edgecolor='red', linewidth=2)
    #     plt.gca().add_patch(rect)
    # plt.show()
    # jdklf
    
    # Extract the centroids (x, y coordinates) of the detected blobs
    blob_centroids = [(int(prop.centroid[0]), int(prop.centroid[1])) for prop in blob_properties]
    blob_areas = [int(prop.area) for prop in blob_properties]
    blob_densities = [blob.area / blob.bbox_area for blob in blob_properties]

    # threshold centroids by area
    noise_pixel_thresh = binar_matrices[image_key][5]
    blob_centroids_thresholded = [i[0] for i in zip(blob_centroids, np.array(blob_areas) > noise_pixel_thresh) if i[1]]
    blob_densities_thresholded = [i[0] for i in zip(blob_densities, np.array(blob_areas) > noise_pixel_thresh) if i[1]]

    
    
    # plot centroids

    ig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(original_matrix)
    
    for centroid in blob_centroids_thresholded:
        ax.plot(centroid[1], centroid[0], 'ro', markersize=0.1)  # Note the (y, x) order for plotting
        
    ax.set_title(image_key)
    plt.show()
    
    
    
    gooby = plt.imshow(image[2000:4000,2000:4000])
    big_centroid_df = pd.DataFrame(blob_centroids_thresholded, columns=['X','Y'])
    #big_centroid_df['density'] = blob_densities_thresholded
    
    for centroid in big_centroid_df[(big_centroid_df['X'] > 2000) & (big_centroid_df['X'] < 4000) & (big_centroid_df['Y'] > 2000) & (big_centroid_df['Y'] < 4000)].index:
        row = big_centroid_df.iloc[centroid]
        
        plt.plot(row['Y'] - 2000, row['X'] - 2000, 'ro', markersize=1)  # Note the (y, x) order for plotting
    plt.title(image_key)
    plt.show()
    
    big_griddles[id] = [blob_centroids_thresholded, image, GM_WM_mask, structure_name, filenum]

    

## assign GM/WM

In [ ]:
import geopandas as gpd
import geojson
from shapely.geometry import shape, Point
import rasterio.features
import skimage
from scipy.ndimage import binary_dilation

filenum_contour_dict = {}


def find_department(point):
    for index, row in df.iterrows():
        if not row.geometry.is_valid:
            row.geometry = row.geometry.buffer(0)
        if row.geometry.contains(point):
            return row['classification']['name']

big_df_dict = {'spatial_X':[], 'spatial_Y':[], 'Parent':[], 'filenum':[], 'phenotype':[]}

for griddlegroup_name in big_griddles:
    contourcount = 0

    print(griddlegroup_name)
    griddlegroup = big_griddles[griddlegroup_name]
    griddle = griddlegroup[0]
    filenum = griddlegroup[4]
    image = big_griddles[griddlegroup_name][1]
    GM_WM_mask = griddlegroup[2]

    print(GM_WM_mask)
    structure_name = griddlegroup[3]
    
    
    with open(GM_WM_mask, encoding='UTF-8') as json_file:
        geojson_f = geojson.load(json_file)
        df = gpd.GeoDataFrame.from_features(geojson_f["features"])
        griddleparents = [find_department(Point(i[1],i[0])) for i in griddle]
    big_df_dict['spatial_X'].extend([i[1] for i in griddle])
    big_df_dict['spatial_Y'].extend([i[0] for i in griddle])
    big_df_dict['Parent'].extend(griddleparents)
    big_df_dict['filenum'].extend([filenum]*len(griddle))
    big_df_dict['phenotype'].extend([structure_name]*len(griddle))
    
    
    
        
        
        
    
    
    
    
        
        
non_nuc_df = pd.DataFrame(big_df_dict)        
    

In [ ]:
# non_nuc_df_onlyabeta = non_nuc_df[non_nuc_df.phenotype == 'aBeta']

# for image_key in binar_matrices.keys():
#     image = binar_matrices[image_key][0]
#     print(image_key)
#     filenum = binar_matrices[image_key][4]
#     structure_name = binar_matrices[image_key][2]

    
    
#     if structure_name == 'aBeta':
#         non_nuc_df_onlyabeta_sub = non_nuc_df_onlyabeta[non_nuc_df_onlyabeta.filenum == filenum]
        
#         gooby = plt.imshow(image[2000:4000,2000:4000])
        
#         non_nuc_df_onlyabeta_sub_sub = non_nuc_df_onlyabeta_sub[(non_nuc_df_onlyabeta_sub['spatial_X'] > 2000) & (non_nuc_df_onlyabeta_sub['spatial_X'] < 4000) & (non_nuc_df_onlyabeta_sub['spatial_Y'] > 2000) & (non_nuc_df_onlyabeta_sub['spatial_Y'] < 4000)]
#         non_nuc_df_onlyabeta_sub_sub['normalized_density'] = non_nuc_df_onlyabeta_sub_sub['density']/np.max(non_nuc_df_onlyabeta_sub_sub['density'])
        
#         plt.scatter(non_nuc_df_onlyabeta_sub_sub['spatial_X'] - 2000, non_nuc_df_onlyabeta_sub_sub['spatial_Y'] - 2000, c = non_nuc_df_onlyabeta_sub_sub.normalized_density, cmap = 'viridis', edgecolor='red') 
        
        
#         plt.show()
    

In [ ]:


# merge data frames

allcells_metadata = adata.obs

allcells_metadata_sub = allcells_metadata[['spatial_X', 'spatial_Y', 'Parent', 'ImageIDOLD', 'phenotype_translated', 'cell', 'marker_intensity_csv_filename']]
allcells_metadata_sub.columns = ['spatial_X', 'spatial_Y', 'Parent', 'filenum', 'phenotype', 'orig_cellid', 'orig_filename']


merged_df = pd.concat([allcells_metadata_sub, non_nuc_df])


ad_cnt_dict = {
'3196':'AD',
'3155':'AD',
'2997':'AD',
'3280':'AD',
'1873':'CNT',
'3628':'CNT',
'3026':'CNT',
'1796':'CNT'
}

merged_df['disease'] = [ad_cnt_dict[i] for i in merged_df['filenum']]



In [ ]:
occurance_dict = dict(collections.Counter(merged_df.phenotype))

df = pd.DataFrame({'phenotype':occurance_dict.keys(), 'count':occurance_dict.values()})

df.plot(x='phenotype', y='count', kind='bar', legend=False)


## add distances to border zones

In [ ]:
import geopandas as gpd
import geojson
from shapely.geometry import shape, Point
import rasterio.features
import skimage
from scipy.ndimage import binary_dilation
import math
from scipy.spatial.distance import cdist


def shortest_distance(coord_set_a, coord_set_b):
    distances = cdist(coord_set_a, coord_set_b)
    shortest_distances = distances.min(axis=1)
    return shortest_distances.tolist()
def remove_border(image, pixels):
    height, width = image.shape
    # Define the region to keep
    # Crop the image
    cropped_image = image[pixels:height-pixels, pixels:width-pixels]
    return cropped_image

filenums = ['3026', '3155', '3280', '3628', '1873', '1796', '3196', '2997']

image_files = ['../image_data/11092021_BaharehNP_3026CtlCorte.qptiff', '../image_data/11162021_Bahareh-3155_EDTA.qptiff', '../image_data/12032021-Bahareh_3280_Alz.qptiff', '../image_data/12252021_Bahareh-3628-Cntr.qptiff', '../image_data/10222021_BaharehNP_1873_CtrlCortex.qptiff', '../image_data/10122021_BaharehNP_1-CtlCortex1796_EDTA_MKT6.qptiff', '../image_data/10122021_BaharehNP_ADCortex_EDTA_3196.qptiff', '../image_data/10202021_BaharehNP_2997-ADCortex_EDTA.qptiff']
GMWM_mask_files = ['../GM_WM_Annotations_QP/3026_GM-WM Annotations.geojson', '../GM_WM_Annotations_QP/3155_GM-WM Annotations.geojson', '../GM_WM_Annotations_QP/3280_GM-WM Annotations.geojson', '../GM_WM_Annotations_QP/3628_GM-WM Annotations.geojson', '../GM_WM_Annotations_QP/1873_GM-WM Annotations.geojson', '../GM_WM_Annotations_QP/1796_GM-WM Annotations.geojson', '../GM_WM_Annotations_QP/3196_GM-WM Annotations.geojson', '../GM_WM_Annotations_QP/2997_GM-WM Annotations.geojson']
GMWM_mask_dict = dict(zip(filenums, GMWM_mask_files))
GM_border_contours = [[4,5],[28], [2], [9,13,14], [3,4], [1,8], [1], [1,2,7]]
WM_border_contours = [[7],[1,2,3,4,7], [1,3,4], [1,2], [6], [3,4], [13], [8]]

cellids = []
distances_to_GM = []
distances_to_WM = []

GM_contour_list = []

for i,filenum in enumerate(filenums):
    print(filenum)
    contourcount = 1
    GM_border_coords = []
    WM_border_coords = []
    sub_df = merged_df[merged_df['filenum'] == filenum]
    GM_WM_mask = GMWM_mask_dict[filenum]
    (all_pages,shapes) = tiff_in(image_files[i])
    
    
    with open(GM_WM_mask, encoding='UTF-8') as json_file:
        geojson_f = geojson.load(json_file)
        df = gpd.GeoDataFrame.from_features(geojson_f["features"])
    
    
    for geom in df['geometry']:
        img = rasterio.features.rasterize([geom], out_shape=all_pages[0][0].shape)
        img = remove_border(img, 50)
        plt.imshow(img)
        plt.show()
        
        
        
        contours = measure.find_contours(img, 0.5)

        # Extract coordinates of contours
        contour_coordinates = [contour.astype(int) for contour in contours]
        
        GM_contours = GM_border_contours[i]
        WM_contours = WM_border_contours[i]
        
        
        
        # Plot contours on top of the binary image
        for j,contour in enumerate(contours):
            plt.imshow(img, cmap='gray')

            plt.plot(contour[:, 1], contour[:, 0], linewidth=2)
            plt.title(contourcount)
            plt.show()
            
            if contourcount in GM_contours:
                contour_coordinates = contour.astype(int)
                GM_border_coords += contour_coordinates.tolist()
            if contourcount in WM_contours:
                contour_coordinates = contour.astype(int)
                WM_border_coords += contour_coordinates.tolist()
            
            contourcount += 1        
        
    
    cell_coords = list(zip(sub_df['spatial_Y'], sub_df['spatial_X']))
    GM_border_distances  = shortest_distance(cell_coords, GM_border_coords)
    WM_border_distances  = shortest_distance(cell_coords, WM_border_coords)
    
    cellids += sub_df.index.tolist()
    distances_to_GM += GM_border_distances
    distances_to_WM += WM_border_distances
    
        
    
GM_didict = dict(zip(cellids, distances_to_GM))
WM_didict = dict(zip(cellids, distances_to_WM))

merged_df['distances_to_outer_edge'] = [GM_didict[i] for i in merged_df.index]
merged_df['distances_to_GM_WM_border'] = [WM_didict[i] for i in merged_df.index]



images_to_view = list(set(merged_df.filenum))

for image in images_to_view:
    sub_merge = merged_df[merged_df['filenum'] == image]
    
    sns.scatterplot(data=sub_merge,x='spatial_X',y='spatial_Y',hue='distances_to_outer_edge', s=2)
    plt.title(image)
    plt.show()
    
    sns.scatterplot(data=sub_merge,x='spatial_X',y='spatial_Y',hue='distances_to_GM_WM_border', s=2)
    plt.title(image)
    plt.show()
    
    
    


## plot merged data

In [ ]:

images_to_view = list(set(merged_df.filenum))

for image in images_to_view:
    sub_merge = merged_df[merged_df['filenum'] == image]
    
    
    
    
#     # Automatically assign colors to unique categories
#     unique_categories = sub_merge['phenotype'].unique()
#     palette = sns.color_palette("husl", len(unique_categories))
#     color_dict = dict(zip(unique_categories, palette))

#     # Create a scatter plot
#     sns.scatterplot(data=sub_merge, x='spatial_X', y='spatial_Y', hue='phenotype', palette=color_dict, s=100, alpha=0.7, size=0)

#     # Set axis labels and title
#     plt.xlabel('X-axis')
#     plt.ylabel('Y-axis')
#     plt.title('Scatter Plot with Points Colored by Category')

#     # Show the legend
#     plt.legend(title='Category', bbox_to_anchor=(1.04, 1), loc='upper left')

#     # Show the plot
#     plt.tight_layout()
#     plt.show()
    
    
    
    fig,ax=plt.subplots(1,1,figsize=(10,8))
    sns.set_style('white')
    ss = sub_merge
    print(len(set(ss['phenotype'])))
    sns.scatterplot(data=ss,x='spatial_X',y='spatial_Y',hue='phenotype', 
                    palette=get_colors(len(set(ss['phenotype']))), s=2)
    ax.invert_yaxis()
    plt.axis('off')
    ax.legend(title='')
    plt.tight_layout()
    plt.title(image)
    plt.show()
    
    
    
    fig,ax=plt.subplots(1,1,figsize=(10,8))
    sns.set_style('white')
    ss = sub_merge

    sns.scatterplot(data=ss,x='spatial_X',y='spatial_Y',hue='Parent', 
                    palette=get_colors(len(set(ss['Parent'])) - 2), s=2)
    ax.invert_yaxis()
    plt.axis('off')
    ax.legend(title='')
    plt.tight_layout()
    plt.show()
    
    
    

In [ ]:
# create additional dataframes for each parent

merged_df_WM = merged_df[merged_df['Parent'] == 'White Matter']
merged_df_GM = merged_df[merged_df['Parent'] == 'Grey Matter']

# merged_df_2_WM = merged_df_2[merged_df_2['Parent'] == 'White Matter']
# merged_df_2_GM = merged_df_2[merged_df_2['Parent'] == 'Grey Matter']



# merged_df_WM_noOTSM = merged_df_noOTSM[merged_df_noOTSM['Parent'] == 'White Matter']
# merged_df_GM_noOTSM = merged_df_noOTSM[merged_df_noOTSM['Parent'] == 'Grey Matter']

# merged_df_2_WM_noOTSM = merged_df_2_noOTSM[merged_df_2_noOTSM['Parent'] == 'White Matter']
# merged_df_2_GM_noOTSM = merged_df_2_noOTSM[merged_df_2_noOTSM['Parent'] == 'Grey Matter']

In [ ]:
# clean up data frames 

dfs = [merged_df, merged_df_WM, merged_df_GM]
df_names = ['merged_df_centroid.pkl', 'merged_df_WM_centroid.pkl', 'merged_df_GM_centroid.pkl']


for i, df in enumerate(dfs):
    print(collections.Counter(df['Parent']))
    df.index = range(len(df))
    df.to_pickle(df_names[i])
    
    


df_to_vis = merged_df

for file in set(df_to_vis['filenum']):
    subdf = df_to_vis[df_to_vis['filenum'] == file]
    for cell in set(subdf['phenotype']):
    
        cellsub_df = subdf[subdf['phenotype'] == cell]

        fig,ax=plt.subplots(1,1,figsize=(10,8))
        sns.set_style('white')
        ss = cellsub_df

        sns.scatterplot(data=ss,x='spatial_X',y='spatial_Y',hue='phenotype', 
                        palette=get_colors(len(set(ss['phenotype']))), s=2)
        ax.invert_yaxis()
        ax.legend(title=cell)
        plt.tight_layout()
        plt.show()
